In [1]:
# 1. Setup: paths and column groups
from pathlib import Path
import numpy as np
import pandas as pd

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = BASE_DIR / "data" / "processed"
SRC = BASE_DIR / "data" / "raw" / "database_row.xlsx"
FILLED_PATH = PROCESSED_DIR / "database_filled.xlsx"
STATS_PATH = BASE_DIR / "outputs" / "tables" / "data_audit_statistics.xlsx"
for p in (FILLED_PATH, STATS_PATH):
    p.parent.mkdir(parents=True, exist_ok=True)

ELEMENTS = ['Ni', 'Al', 'Co', 'Cr', 'Cu', 'Fe', 'Ti',
            'V', 'Mo', 'Hf', 'W', 'Ta', 'B', 'C', 'Nb', 'Mn']
ZERO_COLS = ["HT 1, °C", "HT 1, min", "Recrystallization, °C", "Recrystallization, min",
             "Aging 1, °C", "Aging 1, min", "Aging 2, °C", "Aging 2, min",
             "Cold deformation, %"]
NAN_COLS_NUM = ["Solvus Temperature, °C", "L12 fraction, %"]
NAN_COLS_TXT = ["Cooling_after_HT1", "Cooling_after_Recryst", "Cooling_after_Aging1"]
NAN_COLS = NAN_COLS_NUM + NAN_COLS_TXT

# Raise an error on L12 inconsistencies (True) or only report them (False)
STRICT_L12_CHECK = False


In [2]:
# 2. Loading and column check
df = pd.read_excel(SRC)
df.columns = [c.strip() for c in df.columns]
needed = (ELEMENTS + ZERO_COLS + NAN_COLS +
          ['Paper', 'DOI', 'Phase composition', 'Target state', 'L12 presence'])
missing = [c for c in needed if c not in df.columns]
assert not missing, f"Missing columns: {missing}\nAvailable: {list(df.columns)}"

df = df.replace(r"^\s*$", np.nan, regex=True)


In [3]:
# 3. Validation of phase labels (cleaning was done in the database itself)
ALLOWED_GREEK = 'ητσ'

phase = df['Phase composition']
assert phase.notna().all(), "There are records without phase composition"
assert not phase.str.contains(rf'[^\x00-\x7F{ALLOWED_GREEK}]').any(), \
    "There are characters outside ASCII and the allowed Greek letters"
assert (phase == phase.str.strip()).all(), "There are leading/trailing spaces in labels"
assert not phase.str.contains(r'\s{2,}|(?<! )\+|\+(?! )').any(), "The ' + ' separator format is violated"

tokens = phase.str.split(' + ', regex=False).explode()
print(f"Unique phase combinations: {phase.nunique()}, individual phases: {tokens.nunique()}")
print(tokens.value_counts())


Unique phase combinations: 32, individual phases: 17
Phase composition
FCC              360
L12              282
B2                55
L21               43
η                 26
BCC               18
IM                18
σ                  8
D019               3
Laves              2
Cr23C6             1
other              1
Al-rich phase      1
carbides           1
D022               1
τ                  1
borides            1
Name: count, dtype: int64


In [4]:
# 4. Numeric validation and statistics BEFORE filling
el_num = df[ELEMENTS].apply(pd.to_numeric, errors='coerce')
bad = df[ELEMENTS].notna() & el_num.isna()
assert not bad.any().any(), \
    f"Non-numeric element values: { {el: list(df.index[bad[el]]) for el in bad.columns[bad.any()]} }"

pr_num = df[ZERO_COLS].apply(pd.to_numeric, errors='coerce')
assert not (df[ZERO_COLS].notna() & pr_num.isna()).any().any(), "Non-numeric values in heat-treatment parameters"
opt_num = df[NAN_COLS].apply(pd.to_numeric, errors='coerce')

summary = pd.DataFrame({
    'Metric': ['Records', 'Columns', 'Unique papers', 'Unique DOI'],
    'Value': [len(df), df.shape[1], df['Paper'].nunique(), df['DOI'].nunique()]})

cols_summary = pd.DataFrame({
    'Column': df.columns,
    'dtype': [str(df[c].dtype) for c in df.columns],
    'Filled': [int(df[c].notna().sum()) for c in df.columns],
    'Empty': [int(df[c].isna().sum()) for c in df.columns],
    'Filled, %': [round(df[c].notna().mean() * 100, 1) for c in df.columns],
    'Unique values': [int(df[c].nunique()) for c in df.columns]})

cols_summary = cols_summary[~cols_summary['Column'].isin(ELEMENTS)].reset_index(drop=True)

el_stats = pd.DataFrame({
    'Records with element (>0)': (el_num > 0).sum(),
    'Zero (element absent)': (el_num == 0).sum(),
    'Min (>0), at.%': el_num.where(el_num > 0).min(),
    'Median (>0), at.%': el_num.where(el_num > 0).median(),
    'Max, at.%': el_num.max()}).round(2)
el_stats['Share of records, %'] = (el_stats['Records with element (>0)'] / len(df) * 100).round(1)

proc_stats = pd.DataFrame({
    'Value > 0 (step applied)': (pr_num > 0).sum(),
    'Explicit 0': (pr_num == 0).sum(),
    'Empty (set to 0)': pr_num.isna().sum()})

opt_stats = pd.DataFrame({
    'Reported': df[NAN_COLS].notna().sum(),
    'Not reported (NaN)': df[NAN_COLS].isna().sum(),
    'Zero values': (opt_num == 0).sum()})
opt_stats['Reported, %'] = (opt_stats['Reported'] / len(df) * 100).round(1)

cooling_vc = (pd.concat({c: df[c].astype(object).fillna('NaN').astype(str).value_counts()
                         for c in NAN_COLS_TXT}, axis=1)
              .fillna(0).astype(int).rename_axis('Value').reset_index())

phase_combo = df['Phase composition'].value_counts().rename_axis('Phase composition').reset_index(name='Records')
phase_tokens = (df['Phase composition'].str.split(' + ', regex=False).explode()
                .value_counts().rename_axis('Phase').reset_index(name='Records'))
flags = (pd.DataFrame({'Target state': df['Target state'].value_counts(dropna=False),
                       'L12 presence': df['L12 presence'].value_counts(dropna=False)})
         .fillna(0).astype(int).rename_axis('Value').reset_index())


In [5]:
# 5. Filling (missing processing steps -> 0) and concentration-sum check
df[ELEMENTS] = el_num
df[ZERO_COLS] = pr_num.fillna(0)
assert not df[ELEMENTS + ZERO_COLS].isna().any().any()

TOLERANCE = 0.5
df['computed_sum'] = df[ELEMENTS].sum(axis=1)
dev = (df['computed_sum'] - 100.0).abs()
print(f"Rows with |sum - 100| > {TOLERANCE}: {int((dev > TOLERANCE).sum())}; "
      f"maximum deviation: {dev.max():.3f} at.%")
if 'Sum' in df.columns:
    print(f"max |Sum from database - computed_sum| = "
          f"{(pd.to_numeric(df['Sum'], errors='coerce') - df['computed_sum']).abs().max():.3f}")

# File with visible "NaN" text (true NaN values are kept in memory)
df_out = df.drop(columns=['computed_sum']).copy()
df_out[NAN_COLS] = df_out[NAN_COLS].astype(object).fillna("NaN")
df_out.to_excel(FILLED_PATH, index=False)
print(f"Saved: {FILLED_PATH} | {df_out.shape[0]} rows, {df_out.shape[1]} columns")


Rows with |sum - 100| > 0.5: 0; maximum deviation: 0.200 at.%
max |Sum from database - computed_sum| = 0.000
Saved: C:\Users\Николай\PycharmProjects\FeNiCoAlCrTi-dataset\data\processed\database_filled.xlsx | 374 rows, 38 columns


In [6]:
# 6. Consistency check: L12 fraction vs L12 presence
l12_frac = opt_num["L12 fraction, %"]
l12_presence = pd.to_numeric(df["L12 presence"], errors="coerce")
assert not (df["L12 presence"].notna() & l12_presence.isna()).any(), \
    "Non-numeric values in 'L12 presence'"

# Rows with an unreported fraction (NaN) are excluded: comparisons with NaN are False
bad_zero = df[(l12_frac == 0) & (l12_presence == 1)]  # fraction measured as 0, but L12 is marked as present
bad_pos = df[(l12_frac > 0) & (l12_presence == 0)]    # fraction > 0, but L12 is marked as absent

print(f"L12 fraction = 0 with L12 presence = 1: {len(bad_zero)}")
print(f"L12 fraction > 0 with L12 presence = 0: {len(bad_pos)}")
if len(bad_zero):
    print("\nPapers with fraction = 0 and L12 present:")
    print(bad_zero[["Paper", "DOI"]].drop_duplicates().to_string(index=False))
if len(bad_pos):
    print("\nPapers with fraction > 0 and L12 absent:")
    print(bad_pos[["Paper", "DOI"]].drop_duplicates().to_string(index=False))

if STRICT_L12_CHECK:
    assert len(bad_zero) == 0 and len(bad_pos) == 0, "L12 fraction / presence inconsistencies found"


L12 fraction = 0 with L12 presence = 1: 0
L12 fraction > 0 with L12 presence = 0: 0


In [7]:
# 7. Export of statistics for the tables in the paper
sheets = {
    'Summary': summary,
    'Columns': cols_summary,
    'Elements': el_stats.rename_axis('Element').reset_index(),
    'Processing_params': proc_stats.rename_axis('Column').reset_index(),
    'Optional_fields': opt_stats.rename_axis('Column').reset_index(),
    'Cooling_values': cooling_vc,
    'Phase_combinations': phase_combo,
    'Phase_individual': phase_tokens,
    'Target_L12_flags': flags,
}
with pd.ExcelWriter(STATS_PATH, engine='openpyxl') as writer:
    for name, table in sheets.items():
        table.to_excel(writer, sheet_name=name, index=False)
        ws = writer.sheets[name]
        for col_cells in ws.columns:
            width = max(len(str(c.value or '')) for c in col_cells)
            ws.column_dimensions[col_cells[0].column_letter].width = min(max(width + 3, 10), 60)
print(f"Statistics saved: {STATS_PATH}")


Statistics saved: C:\Users\Николай\PycharmProjects\FeNiCoAlCrTi-dataset\outputs\tables\data_audit_statistics.xlsx


In [8]:
# 8. Phase flags based on the set of phases in the label
PHASE_FLAGS = {'FCC': 'FCC', 'L12': 'L12', 'B2': 'B2', 'BCC': 'BCC',
               'eta': 'η', 'sigma': 'σ', 'L21': 'L21', 'Laves': 'Laves'}

phase_sets = df['Phase composition'].str.split(' + ', regex=False).map(frozenset)
known = set(PHASE_FLAGS.values()) | {'IM'}
print("Phases without a dedicated flag:", sorted(set().union(*phase_sets) - known))

for key, token in PHASE_FLAGS.items():
    df[f'phase_{key}'] = phase_sets.map(lambda s, t=token: int(t in s))
FLAG_COLS = [f'phase_{k}' for k in PHASE_FLAGS]

df['Target state'] = pd.to_numeric(df['Target state'], errors='raise').astype(int)
df_full = df.drop(columns=['computed_sum'])


Phases without a dedicated flag: ['Al-rich phase', 'Cr23C6', 'D019', 'D022', 'borides', 'carbides', 'other', 'τ']


In [9]:
# 9. Unique compositions (key: concentrations rounded to 3 decimals)
df_key = df_full.copy()
df_key[ELEMENTS] = df_full[ELEMENTS].round(3)

n_exact = df_full.groupby(ELEMENTS).ngroups
n_rounded = df_key.groupby(ELEMENTS).ngroups
print(f"Unique compositions: exact comparison {n_exact}, after rounding to 3 decimals {n_rounded}")

agg_spec = {
    'Composition, at.%': ('Composition, at.%', 'first'),
    'papers_count': ('DOI', 'nunique'),
    'n_records': ('DOI', 'size'),
    'Target state': ('Target state', 'max'),
    **{c: (c, 'max') for c in FLAG_COLS},
}
df_unique = df_key.groupby(ELEMENTS, as_index=False).agg(**agg_spec)

conflicts = df_key.groupby(ELEMENTS)['Phase composition'].nunique()
print(f"Records: {len(df_full)}, unique compositions: {len(df_unique)}")
print(f"Compositions with different phase composition under different treatments: {int((conflicts > 1).sum())}")


Unique compositions: exact comparison 161, after rounding to 3 decimals 161
Records: 374, unique compositions: 161
Compositions with different phase composition under different treatments: 25


In [10]:
# 10. Saving
df_full.to_parquet(PROCESSED_DIR / "dataset_experimental_full.parquet", index=False)
df_unique.to_parquet(PROCESSED_DIR / "dataset_compositions_unique.parquet", index=False)
print("Saved: dataset_experimental_full.parquet, dataset_compositions_unique.parquet")

Saved: dataset_experimental_full.parquet, dataset_compositions_unique.parquet
